# 🎮 2D游戏素材生成工具 - Google Colab部署

## 特点
- ✅ 免费T4 GPU (生成速度: 2-3秒/张)
- ✅ 包含七牛云风格缓存功能
- ✅ 支持Gradio公网访问

## 使用说明
1. 点击菜单: 运行时 → 更改运行时类型 → 选择 GPU (T4)
2. 按顺序运行下面的代码块
3. 等待最后一个单元格输出Gradio链接
4. 点击链接即可使用

In [ ]:
# 步骤1: 检查并克隆项目代码
import os

# 检查项目是否已存在
if os.path.exists('2d-game-assets-generator'):
    print("✅ 项目已存在,跳过克隆")
    %cd 2d-game-assets-generator
else:
    print("📥 克隆项目...")
    !git clone https://github.com/lsthaha/2d-game-assets-generator.git
    %cd 2d-game-assets-generator
    print("✅ 克隆完成")

# 验证目录结构
!ls -la
!ls -la backend/ frontend/

In [ ]:
# 步骤2: 安装依赖 (约3-5分钟)
!pip install -q torch torchvision diffusers transformers gradio fastapi uvicorn \
    pillow opencv-python pyyaml python-dotenv peft safetensors requests pydantic qiniu tqdm

In [ ]:
# 步骤3: 配置为GPU模式
config_content = '''
# GPU配置
MODEL_CONFIG = {
    "base_model": "runwayml/stable-diffusion-v1-5",
    "lcm_lora": "latent-consistency/lcm-lora-sdv1-5",
    "device": "cuda",  # Colab GPU
    "dtype": "float16",  # GPU加速
    "enable_attention_slicing": True,
    "enable_memory_efficient_attention": True,
    "background_removal_method": "rembg",
}
'''

# 修改config.py
!sed -i 's/"device": "cpu"/"device": "cuda"/' config.py
!sed -i 's/"dtype": "float32"/"dtype": "float16"/' config.py
!sed -i 's/"enable_memory_efficient_attention": False/"enable_memory_efficient_attention": True/' config.py

print("✅ GPU配置完成")

In [ ]:
# 步骤4: 启动后端 (后台运行)
import subprocess
import time

# 后台启动后端
backend_process = subprocess.Popen(
    ['python', 'backend/main.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print("⏳ 启动后端服务...")
time.sleep(15)  # 等待后端启动
print("✅ 后端已启动在 http://127.0.0.1:8001")

In [ ]:
# 步骤5: 修改并启动前端 (自动生成公网链接)
import os

# 检查文件是否存在
if not os.path.exists('frontend/app.py'):
    print("❌ 错误: frontend/app.py 不存在!")
    print("当前目录:", os.getcwd())
    !ls -la
else:
    print("✅ 找到 frontend/app.py")
    
    # 修改为公网模式
    with open('frontend/app.py', 'r', encoding='utf-8') as f:
        content = f.read()
    
    # 修改launch参数
    if 'share=True' not in content:
        content = content.replace(
            'demo.launch(',
            'demo.launch(share=True, '
        )
        
        with open('frontend/app.py', 'w', encoding='utf-8') as f:
            f.write(content)
        print("✅ 已启用公网分享模式")
    
    # 启动前端
    print("\n🚀 启动前端...")
    print("⏳ 等待生成公网链接 (约10-20秒)")
    print("📌 链接生成后,点击即可访问!\n")
    !python frontend/app.py

## 🎯 使用提示

### 性能对比
- **GPU (T4)**: 2-3秒/张 ⚡
- **CPU**: 2-4分钟/张 🐢

### 推荐配置
- 推理步数: 4-8步
- 引导强度: 7.5
- 使用种子锁定保持风格一致

### 七牛云功能
如需启用七牛云集成:
1. 在Colab中设置环境变量
2. 生成的素材会自动上传到Kodo
3. 风格缓存会同步到云端

```python
import os
os.environ['QINIU_ACCESS_KEY'] = 'your_key'
os.environ['QINIU_SECRET_KEY'] = 'your_secret'
os.environ['QINIU_BUCKET'] = 'your_bucket'
```

## 📊 风格缓存功能测试

测试风格缓存命中率:

In [ ]:
# 测试风格缓存
from backend.integrations import StyleCacheManager, generate_style_cache_key
from pathlib import Path

# 创建缓存管理器
manager = StyleCacheManager(cache_dir=Path("cache"))

# 测试配置
style_config = {
    "美术风格": "pixel_art",
    "色彩主调": "warm_tones",
    "固定种子": 42,
    "像素尺寸": "16px",
    "线条粗细": "medium",
    "饱和度": 75,
    "素材类型": "character"
}

# 生成缓存键
cache_key = generate_style_cache_key(style_config)
print(f"缓存键: {cache_key}")

# 预测命中率
hit_rate, suggestion = manager.calculate_hit_rate_prediction(style_config)
print(f"\n预测命中率: {hit_rate*100:.1f}%")
print(f"建议: {suggestion}")

# 检查缓存
is_hit, level, data = manager.check_cache(style_config)
if is_hit:
    print(f"\n✅ 缓存命中! 级别: {level}")
else:
    print(f"\n❌ 缓存未命中,需要生成新风格")